In [ ]:
class SparkLogisticRegressionFullModel:

    def __init__(self, df):
        self.df = df
        self.model = None
        self.predictions = None
        self.metrics = {}
        print("Class Initialized Successfully")

    # ==========================================
    # Build Features Pipeline
    # ==========================================
    def build_pipeline(self):

        if "features" in self.df.columns:
            self.df = self.df.drop("features")

        numeric_cols = [
            "Age",
            "Income",
            "LoanAmount",
            "Credit_Score",
            "Employment_Years",
            "Credit_History",
            "Has_Defaulted",
            "Dependents"
        ]

        categorical_encoded_cols = [
            "Gender_encoded",
            "Education_Level_encoded",
            "Married_encoded",
            "Job_Type_encoded",
            "Property_Area_encoded"
        ]

        assembler = VectorAssembler(
            inputCols=numeric_cols + categorical_encoded_cols,
            outputCol="features",
            handleInvalid="keep"
        )

        pipeline = Pipeline(stages=[assembler])
        pipeline_model = pipeline.fit(self.df)

        self.df = pipeline_model.transform(self.df)

        print("Pipeline Built Successfully")
        print("Total Rows:", self.df.count())

        self.df = self.df.withColumnRenamed("Loan_Status", "label")
        print("Renamed 'Loan_Status' to 'label'.")

    # ==========================================
    # Split Data
    # ==========================================
    def split_data(self):

        self.train_data, self.test_data = self.df.randomSplit([0.8, 0.2], seed=42)

        print("Train Count:", self.train_data.count())
        print("Test Count:", self.test_data.count())

    # ==========================================
    # Train Model
    # ==========================================
    def train(self):

        lr = LogisticRegression(
            featuresCol="features",
            labelCol="label",
            maxIter=100,
            regParam=0.01,
            elasticNetParam=0.0
        )

        print("Logistic Regression Model Initialized")

        self.model = lr.fit(self.train_data)

        print("Training Complete")

    # ==========================================
    # Prediction
    # ==========================================
    def predict(self):

        self.predictions = self.model.transform(self.test_data)

        print("Prediction Complete")
        self.predictions.select("label", "prediction", "probability").show(10)

    # ==========================================
    # Evaluation
    # ==========================================
    def evaluate(self):

        accuracy = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="accuracy"
        ).evaluate(self.predictions)

        f1 = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="f1"
        ).evaluate(self.predictions)

        precision = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedPrecision"
        ).evaluate(self.predictions)

        recall = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedRecall"
        ).evaluate(self.predictions)

        auc = BinaryClassificationEvaluator(
            labelCol="label",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        ).evaluate(self.predictions)

        print("====== FINAL EVALUATION ======")
        print("Accuracy :", accuracy)
        print("F1 Score :", f1)
        print("Precision:", precision)
        print("Recall   :", recall)
        print("AUC      :", auc)

        self.metrics = {
            "Accuracy": accuracy,
            "F1": f1,
            "Precision": precision,
            "Recall": recall,
            "AUC": auc
        }

    # ==========================================
    # Metrics Visualization
    # ==========================================
    def visualize_metrics(self):

        plt.figure()
        plt.bar(self.metrics.keys(), self.metrics.values())
        plt.title("Evaluation Metrics")
        plt.xlabel("Metrics")
        plt.ylabel("Score")
        plt.show()

    # ==========================================
    # Confusion Matrix
    # ==========================================
    def plot_confusion_matrix(self):

        y_true = self.predictions.select("label").toPandas().squeeze()
        y_pred = self.predictions.select("prediction").toPandas().squeeze()

        cm = confusion_matrix(y_true, y_pred)

        plt.figure()
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title("Confusion Matrix")
        plt.colorbar()

        classes = np.unique(y_true)

        plt.xticks(np.arange(len(classes)), classes)
        plt.yticks(np.arange(len(classes)), classes)

        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")

        for i in range(len(classes)):
            for j in range(len(classes)):
                plt.text(j, i, cm[i, j],
                         ha="center",
                         va="center",
                         color="white" if cm[i, j] > cm.max()/2 else "black")

        plt.show()

    # ==========================================
    # ROC + Precision Recall Curve
    # ==========================================
    def plot_roc_pr(self):

        training_summary = self.model.summary

        # ROC
        roc = training_summary.roc.toPandas()

        plt.figure()
        plt.plot(roc['FPR'], roc['TPR'])
        plt.plot([0,1],[0,1],'--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.show()

        # Precision-Recall
        pr = training_summary.pr.toPandas()

        plt.figure()
        plt.plot(pr['recall'], pr['precision'])
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")
        plt.show()

    # ==========================================
    # Feature Importance (Real Metadata Version)
    # ==========================================
    def feature_importance(self):

        coefficients = self.model.coefficients.toArray()

        feature_metadata_attrs = self.df.schema['features'].metadata['ml_attr']['attrs']

        feature_names = []

        for attr_type in ['numeric', 'binary', 'nominal']:
            if attr_type in feature_metadata_attrs:
                for attr in feature_metadata_attrs[attr_type]:
                    feature_names.append(attr['name'])

        if len(feature_names) != len(coefficients):
            feature_names = [f"feature_{i}" for i in range(len(coefficients))]

        feature_importance = pd.DataFrame({
            "Feature": feature_names,
            "Coefficient": coefficients
        }).sort_values(by="Coefficient", ascending=False)

        print(feature_importance)

        plt.figure()
        plt.barh(feature_importance["Feature"], feature_importance["Coefficient"])
        plt.title("Feature Importance")
        plt.xlabel("Coefficient Value")
        plt.show()

    # ==========================================
    # Save Model
    # ==========================================
    def save_model(self, path="spark_logistic_model"):
        self.model.save(path)
        print("Model Saved Successfully")

# model = SparkLogisticRegressionFullModel(df)
# model.build_pipeline()
# model.split_data()
# model.train()
# model.predict()
# model.evaluate()
# model.visualize_metrics()
# model.plot_confusion_matrix()
# model.plot_roc_pr()
# model.feature_importance()
# model.save_model()
